In [1]:
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langgraph.checkpoint.sqlite import SqliteSaver

# 1. 加载环境变量（在项目 .env 中配置 DEEPSEEK_API_KEY）
load_dotenv()

# 2. 初始化模型，与上一节保持一致
model = init_chat_model(
    "deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}},
)
tools = []

# 3. 使用磁盘文件保存会话；相同数据库路径和 thread_id 对应同一段记忆
# 路径相对于 Notebook 内核的工作目录，打印绝对路径方便查找
# 不要使用 ":memory:"，否则关闭连接后数据就会丢失
# 依赖由项目管理：在项目根目录执行 uv sync
# 单独使用 Notebook 时可先运行 %pip install langgraph-checkpoint-sqlite
db_path = Path("checkpoints.sqlite").resolve()
config = {"configurable": {"thread_id": "sqlite_t1"}}
print(f"会话数据库：{db_path}")

# 4. 首次连接：创建 Agent 并写入会话记忆
# from_conn_string 是上下文管理器，Agent 的操作必须在 with 内完成
# SqliteSaver 会自动创建所需的数据表，退出 with 时关闭数据库连接
with SqliteSaver.from_conn_string(str(db_path)) as checkpointer:
    agent = create_agent(model=model, tools=tools, checkpointer=checkpointer)
    response = agent.invoke(
        {"messages": [HumanMessage(content="你好，我叫虎哥，我最喜欢猫猫。")]},
        config,
    )
    response["messages"][-1].pretty_print()

# 5. 重新连接同一个数据库并重新创建 Agent，验证记忆已保存到磁盘
# 即使重启内核，也可在初始化模型和配置后，单独执行下面这个 with 块恢复会话
# 更换 thread_id 会开启独立会话；重复运行整个单元格会追加新的对话
with SqliteSaver.from_conn_string(str(db_path)) as checkpointer:
    agent = create_agent(model=model, tools=tools, checkpointer=checkpointer)
    saved_state = agent.get_state(config)
    print(f"恢复的历史消息数：{len(saved_state.values.get('messages', []))}")

    response = agent.invoke(
        {"messages": [HumanMessage(content="我最喜欢的动物是什么？")]},
        config,
    )
    for message in response["messages"]:
        message.pretty_print()


会话数据库：D:\AgentProject\LangGraphAgent\LangChain入门笔记\08-LangChain-短期记忆与存储策略\notebools\checkpoints.sqlite
================================== Ai Message ==================================

你好呀虎哥！🐯 猫猫确实超级可爱，毛茸茸软乎乎的，谁能不爱呢～ 你是自己养了猫猫，还是喜欢云吸猫呀？有没有特别喜欢的品种？😺
恢复的历史消息数：2
================================ Human Message =================================

你好，我叫虎哥，我最喜欢猫猫。
================================== Ai Message ==================================

你好呀虎哥！🐯 猫猫确实超级可爱，毛茸茸软乎乎的，谁能不爱呢～ 你是自己养了猫猫，还是喜欢云吸猫呀？有没有特别喜欢的品种？😺
================================ Human Message =================================

我最喜欢的动物是什么？
================================== Ai Message ==================================

哈哈，虎哥，你刚刚才说过你最喜欢猫猫呀！🐱 所以答案当然是——猫猫！对不对？😸 需要我帮你找点猫猫图片或者聊聊猫猫的话题吗？
